# Cadre & objectifs du projet CISIA Santé

**Données synthétiques** · exercice CIF / jury · **jamais** une décision clinique réelle.

Ce notebook pose le **contexte**, les **objectifs** et les **contraintes** avant d’entrer dans les analyses et modèles.

## 1. Contexte

Les établissements hospitaliers font face à une pression croissante sur les capacités d’accueil, au vieillissement de la population et à l’augmentation des pathologies chroniques.

Les **réadmissions non programmées à 30 jours** (retour aux urgences ou nouvelle hospitalisation après une sortie) sont à la fois :

- un **signal clinique** de vulnérabilité du patient ;
- une **charge évitable** pour le système de soins.

Les parcours génèrent des données hétérogènes : SIH, PMSI, biologie, constantes, médicaments, comptes-rendus, historique de soins, objets connectés, proxies territoriaux.

## 2. Mission & objectifs

Mandat : concevoir un système de **prédiction du risque de réadmission à 30 jours** à partir des séjours historiques.

| Objectif | Contenu |
|----------|--------|
| **Analyser** | Distinguer sorties à faible risque vs situations à suivi renforcé |
| **Opérationnaliser** | Solution intégrable au SIH, évolutive (nouvelles sources) |
| **Gouverner** | Qualité des données, biais (âge, territoire, sexe), RGPD art. 9 |

Deux scores pédagogiques dans ce dépôt :

1. **Score sortie** — risque de réadmission 30 j à la sortie d’hospitalisation  
2. **Score télé** — signal de suivi post-sortie (fenêtre objets connectés)

## 3. Cadre réglementaire & éthique

- Données de santé = **données sensibles** (RGPD art. 9) + secret médical  
- **Pas** d’usage de données socio-économiques individuelles non collectées à l’hôpital  
- Proxies territoriaux **agrégés** (INSEE) uniquement  
- Traçabilité des décisions (registre colonnes, justifications)  
- Identité nominative isolée (`data/vault_identite`) — absente des features ML  
- Les modèles **n’excluent pas le jugement soignant**

## 4. Cadre technique du dépôt

| Couche | Rôle |
|--------|------|
| `src/data/` | Pipeline déclarative (registry sources + étapes) |
| `src/model/` | Entraînement, Optuna, inférence, SHAP |
| `src/web/` | Démo CISIA + soignant (bind local) |
| `notebooks/` | Parcours jury DS (inventaire → réponse) |

**Cohorte d’entraînement :** uniquement les séjours avec `ModeSortie == Domicile`  
→ les **décès**, transferts, EHPAD, HAD sont **exclus** (pas de fuite post-mortem).

In [ ]:
from pathlib import Path
import runpy

from IPython.display import Markdown, display

_sp = next(
    a / "setup_path.py"
    for a in [Path.cwd(), *Path.cwd().parents]
    if (a / "setup_path.py").is_file()
)
runpy.run_path(str(_sp))

from _utils.bootstrap import ROOT, paths, PIPELINE_OVERVIEW, SOURCES_OVERVIEW
from src.data.analyse_donnees import cohort_training_summary
from src.data.load import load_csv

display(Markdown(f"**Racine projet :** `{ROOT}`"))
display(Markdown("### Sources déclarées (registry)"))
display(SOURCES_OVERVIEW())
display(Markdown("### Étapes pipeline (registry)"))
display(PIPELINE_OVERVIEW())

sej = load_csv(paths, "sejours.csv", from_raw=False)
coh = cohort_training_summary(sej)
display(Markdown("### Cohorte — exclusion des décès"))
display(Markdown(f"- Règle : **{coh['regle']}**"))
display(Markdown(
    f"- Séjours bruts : **{coh['n_sejours_bruts']}** · "
    f"éligibles : **{coh['n_eligibles_entrainement']}** · "
    f"décès exclus : **{coh['n_deces_exclus']}**"
))


## 5. Suite du parcours

1. [Inventaire des sources](../01_donnees/01_inventaire_sources.ipynb)  
2. [Analyse qualité complète](../01_donnees/05_analyse_donnees.ipynb) ← nulls, sentinelles, dates, patho↔retour  
3. Nettoyage → préparation → entraînement → lancement → réponse (`02_modeles/`)  

Webapp : page **Analyse données** aussi sur `/cisia/analyse`.